# 02 · Campaign — full design campaign on your target + assemble the cohort feature table

**Standard slot:** *design campaign.* **For Project 25 this is two things at once (D2):**
1. run a **complete design campaign** on your advisor-approved target (worked example: a binder
   campaign, the Project 06 pattern) at honest scale, and
2. **assemble the cohort feature table** — the `EXAMPLE_DATA` design→outcome dataset (with a planted
   feature→outcome structure) that the ML success predictor (notebook 04) learns from. In a real
   cohort this table is built from the Projects 01–24 outputs; here it is synthetic and clearly
   labeled.

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster); the
> capstone aggregates a full cohort's compute. **The ML part is light/CPU-OK** (scikit-learn/XGBoost
> on a feature table). The cells below run on the deterministic **mock / EXAMPLE_DATA** path so the
> plumbing executes anywhere; switch to the real backend on Colab. Run `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The capstone uses the **full stack** plus the ML libraries. **Pin commits/tags** and **verify the
URLs still exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin
and log it). This check needs no GPU. Which design tools you verify depends on your `DESIGN_TYPE`.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   ML (always):
#     scikit-learn  https://github.com/scikit-learn/scikit-learn   # success predictor (LogReg/RF) — pin <tag>
#     XGBoost       https://github.com/dmlc/xgboost                 # optional gradient-boosted predictor — pin <tag>
#   Design tools (verify the ones YOUR campaign uses):
#     RFdiffusion   https://github.com/RosettaCommons/RFdiffusion   # backbones / binder mode — pin <commit>
#     BindCraft     https://github.com/martinpacesa/BindCraft       # one-shot binders (A100) — pin <commit>
#     ColabFold     https://github.com/sokrypton/ColabFold          # AF2(-Multimer) — pin <commit>
PINNED = {
    "scikit-learn": "https://github.com/scikit-learn/scikit-learn",
    "XGBoost":      "https://github.com/dmlc/xgboost",
    "RFdiffusion":  "https://github.com/RosettaCommons/RFdiffusion",
    "BindCraft":    "https://github.com/martinpacesa/BindCraft",
    "ColabFold":    "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:13s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:13s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")

## 1 · Run the design campaign on your target (worked example: binder)

Honest campaign sizes (binder example): BindCraft 50–200; RFdiffusion-binder 500–1000 backbones →
ProteinMPNN → AF2-Multimer. We use a small **mock** count so the dry run is fast; scale up with the
real backend on A100. If your `DESIGN_TYPE` is an enzyme/antibody, swap in that family's generator —
the integration pattern is identical. **Mock numbers are SYNTHETIC.**

In [ ]:
import campaign_tools as ct
import pandas as pd

DESIGN_TYPE = "binder"     # keep consistent with notebook 01 (your chosen type)
TARGET = "EXAMPLE_TARGET"
HOTSPOTS = ct.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with your verified residues

N_CAMPAIGN = 200           # -> 500-1000 backbones on A100; small batch / FreeBindCraft on T4
TOOL = "mock"              # -> "bindcraft" / "rfdiffusion" / "rfantibody" / "rfdiffusion2" on Colab (A100)

# Real call (Colab, A100): ct.generate_designs(TARGET, HOTSPOTS, n=N_CAMPAIGN, design_type=DESIGN_TYPE, tool="rfdiffusion")
pool = ct.generate_designs(TARGET, HOTSPOTS, n=N_CAMPAIGN, design_type=DESIGN_TYPE, tool=TOOL)
ct.score_designs(pool, tool=TOOL)              # AF2(-Multimer) -> pae_interaction, plddt, scrmsd, sc
campaign_df = ct.pool_to_df(pool)
campaign_df.to_csv("results/campaign_designs.csv", index=False)
print(f"campaign pool: {len(pool)} designs (design_type={DESIGN_TYPE}, tool={TOOL}; SYNTHETIC if mock)")
print("wrote results/campaign_designs.csv", campaign_df.shape)
campaign_df.head(4)[["design_id", "design_type", "scrmsd", "pae_interaction", "plddt", "rosetta_dG", "synthetic"]]

## 2 · Assemble the cohort feature table (EXAMPLE_DATA)

The ML success predictor learns from the **whole cohort**, not just your campaign. In a real run,
`build_cohort_table(csv_path="...")` loads the table your cohort assembled from Projects 01–24
outputs (sequences, in-silico metrics, any experimental labels). With no real cohort yet, it
**generates a deterministic `EXAMPLE_DATA` cohort** with a planted, *imperfect* feature→outcome
structure so the downstream ML runs anywhere.

We also **fold your own campaign designs into the table** (they get an in-silico-derived
`EXAMPLE_DATA` label here — replace with real experimental labels in notebook 05 when you have them).
Nothing here is a real experimental result.

In [ ]:
import ml_predictor as ml

# Build (or, on a real run, LOAD) the cohort table. Multiple targets/design types => generalization test later.
cohort = ml.build_cohort_table(
    csv_path="data/cohort_design_outcomes.csv",   # if this exists (real cohort), it is LOADED; else synthesized
    n_per_target=120,
    targets=("binder", "binder", "enzyme", "antibody"),  # EXAMPLE cohort composition across Projects 01-24
    seed=0,
)
cohort.to_csv("results/cohort_table.csv", index=False)
print("cohort feature table:", cohort.shape)
print("design types present:", cohort["design_type"].value_counts().to_dict())
print("label origin:", cohort["label_origin"].value_counts().to_dict())
print("overall EXAMPLE_DATA success rate (synthetic!):", round(cohort["success"].mean(), 3))
print("\nEVERY row is source=EXAMPLE_DATA (synthetic) — never report as a real outcome.")
cohort.head(4)

## 3 · (Real cohort) how the table is assembled from Projects 01–24

When you have a real cohort, the table is one row per design with: `design_id`, `design_type`,
`target`, the in-silico features (`scrmsd`, `plddt`, `pae_interaction`, `solubility`, `rosetta_dG`,
`shape_complementarity`, `tm_to_pdb`), and a `success` label. The label is **experimental where wet-lab
data exists** (`label_origin="experimental"`), otherwise an in-silico-derived proxy clearly flagged as
such. See `data/README.md` and `download_data.py` for what to assemble. **Never fabricate
experimental labels.**

In [ ]:
# Scaffold for the REAL assembly (no-op here; the synthetic table above stands in for teaching):
#   frames = []
#   for proj in range(1, 25):
#       p = f"../../project_{proj:02d}_*/results/ranked.csv"   # each project's filtered design table
#       # read, normalize columns to FEATURE_COLUMNS, tag design_type/target, attach any experimental label
#   cohort_real = pd.concat(frames, ignore_index=True)
#   cohort_real.to_csv("data/cohort_design_outcomes.csv", index=False)
print("Real-cohort assembly is a scaffold; the EXAMPLE_DATA table above lets the ML notebook run anywhere.")
print("On a real run, write data/cohort_design_outcomes.csv and re-run cell 2 to LOAD it.")

## D2 checklist
- [ ] Design campaign run at honest scale on your target (mock here; real backend + A100 on Colab).
- [ ] `results/campaign_designs.csv` written (one row per design, metrics parsed, SYNTHETIC if mock).
- [ ] Cohort feature table assembled/loaded → `results/cohort_table.csv` (every synthetic row `EXAMPLE_DATA`).
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the campaign pool.